# Supplier Performance Analysis

## Project Overview
Analyze supplier-level data on delivery timeliness, defect rates, lead times, and cost variance to benchmark suppliers. This is a descriptive analytics project.

## Learning Objectives
- Build supplier scorecards combining multiple performance dimensions
- Rank suppliers using composite performance metrics
- Identify underperformers and cost-risk suppliers
- Analyze performance trends over time

## Problem Statement
Procurement needs a data-driven supplier evaluation: Which suppliers consistently deliver on time with low defects? Which ones have rising costs or worsening lead times?

## Why This Project Matters
Supplier performance directly impacts production schedules, product quality, and costs. Proactive supplier management reduces supply chain risk and improves margins.

## Dataset Overview
Synthetic supplier performance data: ~2,400 monthly records for 20 suppliers over 2 years.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
months = pd.date_range('2023-01-01', '2024-12-01', freq='MS')
n_suppliers = 20
suppliers = [f'SUP-{i:03d}' for i in range(1, n_suppliers + 1)]
categories = np.random.choice(['Raw Materials', 'Components', 'Packaging', 'Services'], n_suppliers)
sup_cat = dict(zip(suppliers, categories))

# Supplier baseline performance (varied quality)
np.random.seed(42)
sup_otd_base = {s: np.clip(np.random.normal(0.88, 0.08), 0.65, 0.99) for s in suppliers}
sup_defect_base = {s: np.clip(np.random.normal(2.0, 1.5), 0.1, 8.0) for s in suppliers}
sup_lead_base = {s: max(3, int(np.random.normal(14, 5))) for s in suppliers}
sup_cost_var = {s: np.random.normal(0, 3) for s in suppliers}

records = []
for m in months:
    for s in suppliers:
        otd = np.clip(sup_otd_base[s] + np.random.normal(0, 0.04), 0.5, 1.0)
        defect = max(0, sup_defect_base[s] + np.random.normal(0, 0.5))
        lead = max(1, int(sup_lead_base[s] + np.random.normal(0, 2)))
        cost_var = sup_cost_var[s] + np.random.normal(0, 1.5)
        orders = max(5, int(np.random.normal(30, 10)))
        
        records.append({
            'Month': m, 'Supplier': s, 'Category': sup_cat[s],
            'OnTimeRate': round(otd, 3), 'DefectRate': round(defect, 2),
            'LeadTimeDays': lead, 'CostVariance': round(cost_var, 2),
            'OrderCount': orders,
        })

df = pd.DataFrame(records)
df['YearMonth'] = df['Month'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Suppliers: {df["Supplier"].nunique()}')
print(f'Avg OTD: {df["OnTimeRate"].mean():.1%}, Avg Defect: {df["DefectRate"].mean():.2f}%')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Month"].min().date()} to {df["Month"].max().date()}')
print(f'\nCategory distribution:\n{pd.Series(sup_cat).value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sup_avg = df.groupby('Supplier')['OnTimeRate'].mean().sort_values()
sup_avg.plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Avg On-Time Rate by Supplier')
axes[0,0].axvline(0.90, color='red', linestyle='--', alpha=0.5, label='90% target')
axes[0,0].legend()

sup_def = df.groupby('Supplier')['DefectRate'].mean().sort_values(ascending=False)
sup_def.plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Avg Defect Rate by Supplier')

cat_otd = df.groupby('Category')['OnTimeRate'].mean().sort_values()
cat_otd.plot.bar(ax=axes[1,0], edgecolor='black', color='green')
axes[1,0].set_title('Avg OTD by Category')
axes[1,0].tick_params(axis='x', rotation=45)

df.groupby('Category')['CostVariance'].mean().plot.bar(ax=axes[1,1], edgecolor='black', color='purple')
axes[1,1].set_title('Avg Cost Variance by Category')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Supplier Scorecard

In [ ]:
scorecard = df.groupby('Supplier').agg(
    category=('Category', 'first'),
    avg_otd=('OnTimeRate', 'mean'),
    avg_defect=('DefectRate', 'mean'),
    avg_lead=('LeadTimeDays', 'mean'),
    avg_cost_var=('CostVariance', 'mean'),
    total_orders=('OrderCount', 'sum'),
).round(3)

# Composite score (higher is better)
scorecard['OTD_Score'] = ((scorecard['avg_otd'] - scorecard['avg_otd'].min()) /
                           (scorecard['avg_otd'].max() - scorecard['avg_otd'].min()) * 100).round(1)
scorecard['Defect_Score'] = ((scorecard['avg_defect'].max() - scorecard['avg_defect']) /
                              (scorecard['avg_defect'].max() - scorecard['avg_defect'].min()) * 100).round(1)
scorecard['Lead_Score'] = ((scorecard['avg_lead'].max() - scorecard['avg_lead']) /
                            (scorecard['avg_lead'].max() - scorecard['avg_lead'].min()) * 100).round(1)
scorecard['Composite'] = ((scorecard['OTD_Score'] * 0.4 + scorecard['Defect_Score'] * 0.35 +
                            scorecard['Lead_Score'] * 0.25)).round(1)
scorecard = scorecard.sort_values('Composite', ascending=False)
print('Supplier Scorecard (Top 10):')
print(scorecard[['category', 'avg_otd', 'avg_defect', 'avg_lead', 'Composite']].head(10))
print('\nBottom 5:')
print(scorecard[['category', 'avg_otd', 'avg_defect', 'avg_lead', 'Composite']].tail(5))

## Composite Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if s >= 70 else 'orange' if s >= 50 else 'red' for s in scorecard['Composite']]
ax.barh(range(len(scorecard)), scorecard['Composite'], color=colors, edgecolor='black')
ax.set_yticks(range(len(scorecard)))
ax.set_yticklabels(scorecard.index, fontsize=9)
ax.set_xlabel('Composite Score')
ax.set_title('Supplier Composite Score (Green ≥70, Orange 50-70, Red <50)')
ax.axvline(70, color='green', linestyle='--', alpha=0.5)
ax.axvline(50, color='orange', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'scorecard.png'), dpi=100, bbox_inches='tight')
plt.show()

## OTD vs Defect Rate Quadrant

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = scorecard
ax.scatter(sc['avg_otd'], sc['avg_defect'], s=sc['total_orders'] / 5, alpha=0.6, edgecolor='black')
for sup in sc.index:
    ax.annotate(sup, (sc.loc[sup, 'avg_otd'], sc.loc[sup, 'avg_defect']), fontsize=7, ha='center', va='bottom')
ax.axhline(sc['avg_defect'].median(), color='gray', linestyle='--', alpha=0.5)
ax.axvline(sc['avg_otd'].median(), color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('On-Time Delivery Rate')
ax.set_ylabel('Defect Rate (%)')
ax.set_title('Supplier Quadrant: OTD vs Defect (bubble = order volume)')
ax.text(0.95, 0.3, 'Best', transform=ax.transAxes, fontsize=12, color='green', ha='right')
ax.text(0.05, 2.5, 'Worst', fontsize=12, color='red')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quadrant.png'), dpi=100, bbox_inches='tight')
plt.show()

## Supplier Performance Trends

In [ ]:
top3 = scorecard.index[:3].tolist()
bot3 = scorecard.index[-3:].tolist()
focus = top3 + bot3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for s in focus:
    sub = df[df['Supplier'] == s]
    color = 'green' if s in top3 else 'red'
    axes[0].plot(sub['Month'], sub['OnTimeRate'], alpha=0.6, label=s, color=color if s == top3[0] or s == bot3[0] else None)
    axes[1].plot(sub['Month'], sub['DefectRate'], alpha=0.6, label=s, color=color if s == top3[0] or s == bot3[0] else None)
axes[0].set_title('OTD Trend — Top 3 vs Bottom 3')
axes[0].legend(fontsize=7)
axes[0].tick_params(axis='x', rotation=45)
axes[1].set_title('Defect Rate Trend — Top 3 vs Bottom 3')
axes[1].legend(fontsize=7)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- The **composite scorecard** clearly separates strong from weak suppliers
- **OTD rate** and **defect rate** are the most actionable dimensions for supplier improvement
- The **quadrant chart** identifies suppliers needing immediate attention (high defect + low OTD)
- **Category-level** performance varies — some categories have inherently harder supply chains
- **Cost variance** highlights suppliers with pricing drift that may need renegotiation
- Top suppliers should be rewarded with volume; bottom suppliers need corrective action plans

## Limitations
- Synthetic data with static supplier baselines
- No financial impact quantification (cost of defects, stockout losses)
- No supplier relationship context (sole source, multi-source)
- No contract or SLA data
- No risk assessment (geographic, financial stability)

## How to Improve This Project
- Use real procurement and quality data
- Add total cost of ownership analysis
- Include supplier risk scoring (financial health, geographic concentration)
- Build automated scorecards with drill-down reporting
- Add correlation with production disruptions

## Production Considerations
- Automated monthly supplier scorecards with trend alerts
- Integration with procurement systems for supplier selection support
- Trigger-based escalation when metrics breach SLA thresholds
- Annual strategic supplier review powered by historical analytics

## Common Mistakes
- Weighting all scorecard dimensions equally without business context
- Comparing suppliers across different categories without normalization
- Ignoring volume — a low-volume supplier's bad numbers have less impact
- Not communicating scorecards to suppliers for improvement collaboration

## Mini Challenge / Exercises
1. Which category has the widest spread between its best and worst supplier on OTD?
2. If you dropped the bottom 3 suppliers and redistributed volume to top 3, what would be the portfolio's new avg defect rate?
3. Create a risk tier: High Risk (Composite < 40), Watch (40-60), Preferred (60-80), Strategic (80+). How many in each?
4. Which supplier improved most from H1 2023 to H2 2024 on OTD?

## Final Summary / Key Takeaways
- Supplier performance analysis enables data-driven procurement decisions
- Composite scoring combines multiple dimensions into a single actionable metric
- Quadrant analysis quickly identifies suppliers needing attention
- Trend monitoring detects improving and deteriorating suppliers early
- Regular benchmarking drives accountability and continuous improvement